# XFakeNet – Week 4b: faithful explanations

Week 4 showed the built-in attention highlights are not faithful. This notebook computes **SHAP** and
**occlusion** explanations from the model's actual behaviour, runs the same faithfulness tests on all
methods side by side, and switches the pipeline to SHAP highlights. **No retraining.**

| Section | What it does | Time with GPU | Without GPU |
|---|---|---|---|
| A | Attention vs occlusion vs SHAP, faithfulness tests + example page | ~10 min (all 1,283 statements) | ~30–40 min (400 statements) |
| B | Re-run the pipeline examples with SHAP highlights | ~3 min | ~5 min |
| C | *Optional:* upload your hand-check answers | seconds | seconds |
| D | Final tables + results zip (also saved to Drive) | ~1 min | ~1 min |

**Steps:** `Runtime → Change runtime type → T4 GPU` (if Colab says no GPU is available, choose
**Connect without GPU**; it still works, just slower), then `Runtime → Run all`, allow Google Drive, keep the tab open.

## 1. Google Drive and Week 4 files

In [ ]:
import os, subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none (CPU mode: uses 400 statements, takes ~30–40 min)")
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/xfakenet"
os.chdir(WORK)
need = ["runs/xfakenet_v2/xfakenet.pt", "data/processed/corpus_test.csv", "results/cscore/cscore_results.json"]
missing = [f for f in need if not os.path.exists(f)]
if missing:
    raise SystemExit(f"Missing files: {missing}")
print("Model and data found.")

## 2. Write the project code

In [ ]:
import os
for d in ['configs','src/xfakenet','scripts','results/week4','results/final','data/deception']:
    os.makedirs(d, exist_ok=True)
print('folders ready')

In [ ]:
%%writefile configs/week1.yaml
seed: 42
paths:
  raw: data/raw
  processed: data/processed
  images: data/images
  runs: runs

liar:
  dir: data/raw/liar            # expects train.tsv / valid.tsv / test.tsv

extra_classes:
  enabled: false      # three-class setup: LIAR only
  source: tshp17
  file: data/raw/tshp17/fulltrain.csv   # columns: label,text (no header)
  test_file: data/raw/tshp17/balancedtest.csv
  use_hoax_trusted: false
  # Articles are long; LIAR statements are ~18 words. Truncate to the first
  # sentences up to this many words so length is not a shortcut for the class.
  max_words: 40
  # Cap per class so Satire/Propaganda stay minority classes (as in the outline).
  max_per_class: 1500

fakenewsnet:
  dir: data/raw/FakeNewsNet/dataset
  sample_size: 1000            # articles to try for images (balanced fake/real)
  min_image_side: 100          # px; smaller images (icons, pixels) are discarded
  workers: 16
  timeout: 10
  use_wayback: true

baseline:
  model: roberta-base
  max_len: 64
  epochs: 4
  batch_size: 32
  lr: 2.0e-5
  warmup_ratio: 0.06
  weight_decay: 0.01
  fp16: true

# ---------------- Week 2 ----------------
augmentation:                  # final training mix (week 3): 70% original / 15% / 15%
  original_share: 70
  textfooler_share: 15
  paraphrase_share: 15

cscore:
  clip_model: openai/clip-vit-base-patch32
  batch_size: 64
  percentile: 95               # alternative threshold: 95th percentile of aligned C-scores (valid)
  out_dir: results/cscore

paraphrase:
  models: [gemini-3.5-flash-lite, gemini-3.8-flash, gemini-3.1-flash-lite, gemini-2.5-flash]
  batch_size: 20               # statements per API call
  temperature: 0.9
  sleep_s: 4                   # pause between calls (free-tier rate limits)
  test_n: 500
  out_train: data/augment/paraphrase_train.jsonl
  out_test: data/augment/paraphrase_test.jsonl

textfooler:
  victim_dir: runs/baseline/best
  embeddings_url: https://github.com/nmrksic/counter-fitting/raw/master/word_vectors/counter-fitted-vectors.txt.zip
  embeddings_path: data/raw/counter-fitted-vectors.txt.zip
  sim_model: sentence-transformers/all-MiniLM-L6-v2
  k: 50                        # synonyms per word (paper: 50)
  sim_threshold: 0.80
  max_change: 0.25             # at most 25% of words replaced
  query_budget: 2000
  test_n: 500
  out_train: data/augment/textfooler_train.jsonl
  out_test: data/augment/textfooler_test.jsonl

model_v1:
  lrs: [1.0e-5, 2.0e-5, 3.0e-5]
  epochs: 5

# ---------------- Week 3 ----------------
xfakenet:                      # RoBERTa + jointly trained token/segment attention
  encoder: roberta-base
  max_len: 64
  max_segments: 8
  att_dim: 256
  epochs: 4
  batch_size: 32
  lr: 2.0e-5                   # encoder
  head_lr: 1.0e-4              # attention layers + classifier
  warmup_ratio: 0.06
  weight_decay: 0.01
  highlight_threshold: 0.65    # outline: tokens above T = 0.65 are highlighted

modality:
  C: 0.1                       # regularisation of the text/image logistic models (389 items)

week3:
  out_dir: results/week3

# ---------------- Week 4 ----------------
week4:
  out_dir: results/week4

deception:
  n_mismatched: 200            # shuffled pairs added to the 389 real ones for labelling
  batch_size: 5                # image-text pairs per Gemini call
  image_side: 512
  sleep_s: 4
  C: 0.5
  min_per_class: 5
  handcheck_n: 100
  labels: data/deception/gemini_labels.jsonl
  features: data/deception/clip_features.npz
  oof: data/deception/classifier_oof.csv
  model: runs/deception_clf.joblib
  handcheck_sample: data/deception/handcheck_sample.csv
  handcheck_results: data/deception/handcheck_results.csv

pipeline:
  modality_model: runs/modality.joblib

explain:                       # faithful explanations (week 4 fix)
  permutations_gpu: 10
  permutations_cpu: 6
  cpu_statements: 400
  chunk: 50

In [ ]:
%%writefile configs/label_mapping.yaml
# XFakeNet five-class label mapping (frozen 2026-10-21, Week 1)
# Every later step (training, augmentation, evaluation) reads this file.
# Change it only with a note in docs/label_mapping.md and a full retrain.

# Three-class setup (changed 20 Oct 2026): Satire/Propaganda dropped because the
# only available source (TSHP-17) taught the model website style, not the class
# (Propaganda F1 0.99 on same-site validation, 0.08 on unseen-site test).
# For the old five-class setup use:
#   classes: [Fabricated, Misleading, Satire, Propaganda, Credible]
#   and set extra_classes.enabled: true in week1.yaml
classes: [Fabricated, Misleading, Credible]   # index order = model label ids

liar:
  # LIAR's six PolitiFact truth ratings -> XFakeNet classes
  # Keys are quoted on purpose: unquoted true/false become YAML booleans.
  "pants-fire":  Fabricated
  "false":       Fabricated
  "barely-true": Misleading
  "half-true":   Misleading
  "mostly-true": Credible
  "true":        Credible

# LIAR has no Satire or Propaganda signal. These two classes come from an
# external labelled source (default: TSHP-17, Rashkin et al. 2017).
tshp17:
  # TSHP-17 integer labels: 1=Satire 2=Hoax 3=Propaganda 4=Trusted
  1: Satire
  3: Propaganda
  # 2 (Hoax) and 4 (Trusted) are NOT used by default: they would add
  # long-article Fabricated/Credible examples whose style differs from LIAR.
  # Set use_hoax_trusted: true in configs/week1.yaml to include them.
  2: Fabricated
  4: Credible

fakenewsnet:
  # Binary only; used for C-score calibration/evaluation, not five-class training.
  "fake": Fabricated
  "real": Credible

In [ ]:
%%writefile src/xfakenet/__init__.py
# package marker

In [ ]:
%%writefile src/xfakenet/common.py
"""Shared helpers: config loading, label mapping, seeding, text cleanup."""
from __future__ import annotations

import json
import random
import re
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[2]


def load_yaml(path: str | Path) -> dict:
    with open(ROOT / path if not Path(path).is_absolute() else path) as f:
        return yaml.safe_load(f)


def load_config() -> dict:
    return load_yaml("configs/week1.yaml")


def load_mapping() -> dict:
    return load_yaml("configs/label_mapping.yaml")


def class_names() -> list[str]:
    return load_mapping()["classes"]


def label2id() -> dict[str, int]:
    return {c: i for i, c in enumerate(class_names())}


def p(rel: str | Path) -> Path:
    """Resolve a repo-relative path."""
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def set_seed(seed: int) -> None:
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


_WS = re.compile(r"\s+")
_SENT = re.compile(r"(?<=[.!?])\s+")


def clean_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.replace("“", '"').replace("”", '"').replace("’", "'")
    return _WS.sub(" ", s).strip()


def truncate_words(text: str, max_words: int) -> str:
    """Keep whole leading sentences up to max_words; hard-cut if the first sentence is longer."""
    text = clean_text(text)
    out, n = [], 0
    for sent in _SENT.split(text):
        w = sent.split()
        if n + len(w) > max_words:
            if not out:
                out.append(" ".join(w[:max_words]))
            break
        out.append(sent)
        n += len(w)
    return " ".join(out)


def write_json(obj, path: str | Path) -> None:
    with open(p(path), "w") as f:
        json.dump(obj, f, indent=2, default=str)

In [ ]:
%%writefile src/xfakenet/segments.py
"""Split a statement into segments (sentences and clauses) for the sentence-attention layer.

About 81% of LIAR statements are a single sentence, so splitting only at sentence
boundaries would give the attention layer nothing to choose between. We also split
at ; : commas and dashes, then merge pieces shorter than `min_words` into their
neighbour so a segment is always a meaningful clause.
"""
from __future__ import annotations

import re

_SPLIT = re.compile(r"(?<=[.!?;:])\s+|,\s+|\s+--?\s+")


def segment_spans(text: str, max_segments: int = 8, min_words: int = 3) -> list[tuple[int, int]]:
    """Return character spans (start, end) of the segments of `text`."""
    spans, start = [], 0
    for m in _SPLIT.finditer(text):
        if m.start() > start:
            spans.append((start, m.start()))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    if not spans:
        return [(0, len(text))]

    words = lambda s: len(text[s[0]:s[1]].split())  # noqa: E731
    merged: list[tuple[int, int]] = []
    for s in spans:
        if merged and (words(s) < min_words or words(merged[-1]) < min_words):
            merged[-1] = (merged[-1][0], s[1])
        else:
            merged.append(s)
    if len(merged) > max_segments:
        merged = merged[: max_segments - 1] + [(merged[max_segments - 1][0], merged[-1][1])]
    return merged


def segments(text: str, max_segments: int = 8, min_words: int = 3) -> list[str]:
    return [text[a:b] for a, b in segment_spans(text, max_segments, min_words)]

In [ ]:
%%writefile src/xfakenet/explain_utils.py
"""Word- and segment-level explanation from attention weights (no torch needed)."""
from __future__ import annotations

from bisect import bisect_right

import numpy as np


def word_explanation(text, offsets, seg_ids, tok_att, seg_att, spans, probs, classes, thr) -> dict:
    """Turn token/segment attention into word-level and segment-level explanations (pure numpy)."""
    words, w_imp, pos = [], [], 0
    for w in text.split():
        a = text.index(w, pos)
        words.append((a, a + len(w), w))
        w_imp.append(0.0)
        pos = a + len(w)
    starts = [a for a, _, _ in words]
    for j, (a, z) in enumerate(offsets):
        if z > a and seg_ids[j] >= 0 and words:
            k = max(0, bisect_right(starts, a) - 1)
            w_imp[k] += float(tok_att[j])      # a word's importance = sum over its sub-word tokens
    m = max(w_imp) if w_imp and max(w_imp) > 0 else 1.0
    norm = [x / m for x in w_imp]
    segs = [{"text": text[a:z], "weight": round(float(seg_att[k]), 4)}
            for k, (a, z) in enumerate(spans) if k < len(seg_att)]
    top = int(np.argmax([s["weight"] for s in segs])) if segs else 0
    return {
        "text": text,
        "pred": classes[int(np.argmax(probs))],
        "probs": {c: round(float(p_), 4) for c, p_ in zip(classes, probs)},
        "words": [{"w": w, "weight": round(x, 4)} for (_, _, w), x in zip(words, norm)],
        "highlighted": [w for (_, _, w), x in zip(words, norm) if x >= thr],
        "segments": segs,
        "top_segment": segs[top]["text"] if segs else text,
    }

In [ ]:
%%writefile src/xfakenet/model.py
"""XFakeNet classifier: RoBERTa + jointly trained token- and segment-attention (Module 1 + 3).

    tokens --RoBERTa--> h_t
    token attention (within each segment):   a_t = softmax_t( v_t . tanh(W_t h_t) )
    segment vector:                          s_k = sum_t a_t h_t
    segment attention:                       alpha_k = softmax_k( v_s . tanh(W_s s_k) )
    statement vector:                        d = sum_k alpha_k s_k
    prediction:                              softmax(W_c d)

The classifier only sees d, so the attention weights are part of the decision
itself (explanation produced in the forward pass), not computed afterwards.
Token importance shown to the user = a_t * alpha_(segment of t).
"""
from __future__ import annotations

import json
from bisect import bisect_right
from pathlib import Path
from typing import Sequence

import numpy as np
import torch
from torch import nn

from .explain_utils import word_explanation
from .segments import segment_spans


class XFakeNetModel(nn.Module):
    def __init__(self, encoder: nn.Module, num_labels: int, max_segments: int = 8,
                 att_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.encoder = encoder
        d = encoder.config.hidden_size
        self.K = max_segments
        self.tok_proj, self.tok_v = nn.Linear(d, att_dim), nn.Linear(att_dim, 1, bias=False)
        self.seg_proj, self.seg_v = nn.Linear(d, att_dim), nn.Linear(att_dim, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.cls = nn.Linear(d, num_labels)

    def forward(self, input_ids, attention_mask, seg_ids):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # B L D
        b, l, _ = h.shape
        e = self.tok_v(torch.tanh(self.tok_proj(h))).squeeze(-1).float()                       # B L
        ks = torch.arange(self.K, device=h.device)
        member = seg_ids.unsqueeze(1) == ks.view(1, -1, 1)                                       # B K L
        scores = e.unsqueeze(1).expand(b, self.K, l).masked_fill(~member, -1e4)
        a = torch.softmax(scores, -1) * member                                                   # B K L (empty segment -> 0)
        s = torch.bmm(a.to(h.dtype), h)                                                          # B K D
        valid = member.any(-1)                                                                   # B K
        u = self.seg_v(torch.tanh(self.seg_proj(s))).squeeze(-1).float().masked_fill(~valid, -1e4)
        alpha = torch.softmax(u, -1) * valid                                                     # B K
        doc = torch.bmm(alpha.unsqueeze(1).to(s.dtype), s).squeeze(1)                            # B D
        logits = self.cls(self.drop(doc)).float()
        token_imp = (a * alpha.unsqueeze(-1)).sum(1)                                             # B L
        return {"logits": logits, "token_att": token_imp, "seg_att": alpha}


def encode(tok, texts: Sequence[str], max_len: int, max_segments: int, device=None):
    """Tokenise and assign every token to a segment (special/pad tokens get -1)."""
    enc = tok(list(texts), truncation=True, max_length=max_len, padding=True,
              return_offsets_mapping=True, return_tensors="pt")
    offsets = enc.pop("offset_mapping")
    seg_ids = torch.full(enc["input_ids"].shape, -1, dtype=torch.long)
    spans_all = []
    for i, text in enumerate(texts):
        spans = segment_spans(text, max_segments)
        starts = [a for a, _ in spans]
        spans_all.append(spans)
        for j, (a, z) in enumerate(offsets[i].tolist()):
            if z > a:
                seg_ids[i, j] = max(0, bisect_right(starts, a) - 1)
    enc["seg_ids"] = seg_ids
    if device is not None:
        enc = {k: v.to(device) for k, v in enc.items()}
    return enc, offsets, spans_all


def build(encoder_name: str, num_labels: int, max_segments: int, att_dim: int) -> XFakeNetModel:
    from transformers import AutoModel
    return XFakeNetModel(AutoModel.from_pretrained(encoder_name), num_labels, max_segments, att_dim)


def save(model: XFakeNetModel, tok, out_dir: str | Path, meta: dict) -> None:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out / "xfakenet.pt")
    model.encoder.config.save_pretrained(out / "encoder_config")
    tok.save_pretrained(out)
    json.dump(meta, open(out / "xfakenet_meta.json", "w"), indent=2)


def load(model_dir: str | Path, device: str = "cpu"):
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    d = Path(model_dir)
    meta = json.load(open(d / "xfakenet_meta.json"))
    enc = AutoModel.from_config(AutoConfig.from_pretrained(d / "encoder_config"))
    model = XFakeNetModel(enc, len(meta["classes"]), meta["max_segments"], meta["att_dim"])
    model.load_state_dict(torch.load(d / "xfakenet.pt", map_location="cpu"))
    return model.to(device).eval(), AutoTokenizer.from_pretrained(d), meta


class Predictor:
    """Batch prediction + explanations for a saved XFakeNet model."""

    def __init__(self, model_dir: str | Path, batch_size: int = 64, device: str | None = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model, self.tok, self.meta = load(model_dir, self.device)
        self.bs = batch_size

    @torch.no_grad()
    def _run(self, texts: Sequence[str]):
        for s in range(0, len(texts), self.bs):
            batch = list(texts[s:s + self.bs])
            enc, offsets, spans = encode(self.tok, batch, self.meta["max_len"], self.meta["max_segments"],
                                         self.device)
            with torch.autocast(device_type="cuda", enabled=self.device == "cuda"):
                out = self.model(**enc)
            yield batch, enc, offsets, spans, out

    def predict_proba(self, texts: Sequence[str]) -> np.ndarray:
        res = [torch.softmax(o["logits"], -1).cpu().numpy() for *_, o in self._run(texts)]
        return np.concatenate(res) if res else np.zeros((0, len(self.meta["classes"])))

    def explain(self, texts: Sequence[str], threshold: float | None = None) -> list[dict]:
        thr = self.meta.get("highlight_threshold", 0.65) if threshold is None else threshold
        out = []
        for batch, enc, offsets, spans, o in self._run(texts):
            probs = torch.softmax(o["logits"], -1).cpu().numpy()
            tok_att = o["token_att"].float().cpu().numpy()
            seg_att = o["seg_att"].float().cpu().numpy()
            seg_ids = enc["seg_ids"].cpu().numpy()
            offs = offsets.tolist()
            for i, text in enumerate(batch):
                out.append(word_explanation(text, offs[i], seg_ids[i], tok_att[i], seg_att[i], spans[i],
                                            probs[i], self.meta["classes"], thr))
        return out

In [ ]:
%%writefile src/xfakenet/attribution.py
"""Faithful (perturbation-based) explanations for any text classifier.

Week 4 showed that the built-in attention weights are not faithful: removing the
highlighted words hurts the prediction barely more than removing random words, and
the attention top segment agrees with SHAP only at chance level. So the explanations
shown to the user are now computed from the model's actual behaviour:

- word_shap     : Shapley values of each word, estimated by permutation sampling
                  (Strumbelj & Kononenko 2014; the estimator behind SHAP's sampling explainer).
                  A word's value = its average change in the predicted-class probability when
                  added to random subsets of the other words.
- occlusion     : leave-one-out: drop in predicted-class probability when the word is deleted.
- segment_shap  : exact Shapley values over segments (all 2^K subsets, K <= 8).

All functions batch every text they need into ONE call to `predict_proba`, so they are
fast on a GPU. Removed words/segments are deleted from the text.
"""
from __future__ import annotations

import itertools
import math
from typing import Callable, Sequence

import numpy as np

Predict = Callable[[Sequence[str]], np.ndarray]


def _join(words: list[str], keep) -> str:
    return " ".join(w for i, w in enumerate(words) if i in keep)


def plan_word_shap(words: list[str], m: int, rng: np.random.Generator):
    """Texts needed for permutation Shapley + bookkeeping to turn their scores into values."""
    n = len(words)
    texts, steps = [""], []          # texts[0] = empty statement (shared start of every chain)
    for _ in range(m):
        perm = rng.permutation(n)
        prev, keep = 0, set()
        for j in perm:
            keep.add(int(j))
            texts.append(_join(words, keep))
            steps.append((int(j), prev, len(texts) - 1))   # value(j) += p[cur] - p[prev]
            prev = len(texts) - 1
    return texts, steps


def finish_word_shap(n: int, m: int, steps, p: np.ndarray) -> np.ndarray:
    phi = np.zeros(n)
    for j, a, b in steps:
        phi[j] += p[b] - p[a]
    return phi / max(1, m)


def plan_occlusion(words: list[str]):
    return [_join(words, set(range(len(words))) - {i}) for i in range(len(words))]


def plan_segment_shap(segs: list[str]):
    k = len(segs)
    subsets = [S for r in range(k + 1) for S in itertools.combinations(range(k), r)]
    return [" ".join(segs[j] for j in S) for S in subsets], subsets


def finish_segment_shap(k: int, subsets, p: np.ndarray) -> np.ndarray:
    v = dict(zip(subsets, p))
    phi = np.zeros(k)
    for i in range(k):
        others = [j for j in range(k) if j != i]
        for r in range(len(others) + 1):
            w = math.factorial(r) * math.factorial(k - r - 1) / math.factorial(k)
            for S in itertools.combinations(others, r):
                phi[i] += w * (v[tuple(sorted(S + (i,)))] - v[S])
    return phi


def explain_batch(predict: Predict, items: list[dict], classes: list[str], m: int = 8,
                  seed: int = 42, methods=("shap", "occlusion", "segment")) -> list[dict]:
    """items: [{"words": [...], "segments": [...], "pred": class name}] -> adds scores per method.

    One predict_proba call for everything (chunk `items` yourself to bound memory)."""
    rng = np.random.default_rng(seed)
    all_texts, plans = [], []
    for it in items:
        words, plan = it["words"], {}
        if "shap" in methods and words:
            t, steps = plan_word_shap(words, m, rng)
            plan["shap"] = (len(all_texts), len(t), steps)
            all_texts += t
        if "occlusion" in methods and words:
            t = plan_occlusion(words)
            plan["occ"] = (len(all_texts), len(t))
            all_texts += t + [" ".join(words)]
        if "segment" in methods and 2 <= len(it["segments"]) <= 8:
            t, subsets = plan_segment_shap(it["segments"])
            plan["seg"] = (len(all_texts), len(t), subsets)
            all_texts += t
        plans.append(plan)
    probs = predict(all_texts) if all_texts else np.zeros((0, len(classes)))
    out = []
    for it, plan in zip(items, plans):
        c = classes.index(it["pred"])
        r = {}
        if "shap" in plan:
            s, n, steps = plan["shap"]
            r["shap"] = finish_word_shap(len(it["words"]), m, steps, probs[s:s + n, c]).tolist()
        if "occ" in plan:
            s, n = plan["occ"]
            full = probs[s + n, c]
            r["occlusion"] = (full - probs[s:s + n, c]).tolist()
        if "seg" in plan:
            s, n, subsets = plan["seg"]
            r["segment_shap"] = finish_segment_shap(len(it["segments"]), subsets, probs[s:s + n, c]).tolist()
        out.append(r)
    return out


def highlight(words: list[str], scores: Sequence[float], thr: float = 0.65) -> list[str]:
    """Words whose positive contribution is >= thr x the largest positive contribution."""
    s = np.asarray(scores, dtype=float)
    top = s.max() if len(s) else 0.0
    if top <= 0:
        return []
    return [w for w, x in zip(words, s) if x > 0 and x / top >= thr]

In [ ]:
%%writefile src/xfakenet/clip_utils.py
"""CLIP ViT-B/32 helpers shared by the C-score, modality flag, deception classifier and pipeline."""
from __future__ import annotations

import numpy as np


class Clip:
    def __init__(self, name: str = "openai/clip-vit-base-patch32", device: str | None = None):
        import torch
        from transformers import CLIPModel, CLIPProcessor
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = CLIPModel.from_pretrained(name).to(self.device).eval()
        self.proc = CLIPProcessor.from_pretrained(name)

    def embed(self, titles: list[str], images: list, batch_size: int = 64):
        """Normalised (text, image) embeddings. `images` are PIL images or file paths."""
        from PIL import Image
        T, I = [], []
        for s in range(0, len(titles), batch_size):
            ims = [Image.open(x).convert("RGB") if isinstance(x, (str, bytes)) or hasattr(x, "__fspath__")
                   else x.convert("RGB") for x in images[s:s + batch_size]]
            inp = self.proc(text=list(titles[s:s + batch_size]), images=ims, return_tensors="pt",
                            padding=True, truncation=True, max_length=77).to(self.device)
            with self.torch.no_grad():
                out = self.model(**inp)
            T.append(self.torch.nn.functional.normalize(out.text_embeds, dim=-1).cpu().numpy())
            I.append(self.torch.nn.functional.normalize(out.image_embeds, dim=-1).cpu().numpy())
        return np.concatenate(T), np.concatenate(I)


def pair_features(T: np.ndarray, I: np.ndarray) -> np.ndarray:
    """Features for the deception-pattern classifier: text, image, their product, and the C-score."""
    c = 1.0 - (T * I).sum(1, keepdims=True)
    return np.hstack([T, I, T * I, c])

In [ ]:
%%writefile src/xfakenet/pipeline.py
"""The integrated XFakeNet pipeline: one call returns class, explanations, modality flag and pattern.

    pipe = XFakeNetPipeline.load()
    out = pipe("Headline or statement text", image="path/or/PIL.Image")   # image optional

Output keys:
  label, probs                      classifier decision (Fabricated / Misleading / Credible)
  highlighted, word_scores,         Module 1 explanations. Highlights and the top segment come from
  segments, top_segment             SHAP (faithful); the built-in attention versions are kept as
  attention_highlighted,            attention_highlighted / attention_top_segment for comparison.
  attention_top_segment
  modality_flag                     TEXT / IMAGE / CROSS-MODAL (TEXT when no image is given)
  cscore, cscore_threshold          Module 2 contradiction score (only with an image)
  deception_pattern, pattern_probs  one of six patterns (only when CROSS-MODAL)
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

from .common import load_config, p


class XFakeNetPipeline:
    def __init__(self, model_dir, cscore_threshold, modality_path=None, deception_path=None,
                 clip_name="openai/clip-vit-base-patch32", shap_permutations: int = 10, threshold: float = 0.65):
        from .model import Predictor
        self.pred = Predictor(model_dir)
        self.thr = float(cscore_threshold)
        self.m, self.hl_thr = shap_permutations, threshold
        self.clip_name = clip_name
        self._clip = None
        self.modality = self._load(modality_path)
        self.deception = self._load(deception_path)

    @staticmethod
    def _load(path):
        if path and Path(path).exists():
            import joblib
            return joblib.load(path)
        return None

    @classmethod
    def load(cls, model_dir: str = "runs/xfakenet_v2"):
        cfg = load_config()
        thr = json.load(open(p(f"{cfg['cscore']['out_dir']}/cscore_results.json")))["chosen_threshold"]
        return cls(p(model_dir), thr, p(cfg["pipeline"]["modality_model"]), p(cfg["deception"]["model"]),
                   cfg["cscore"]["clip_model"], cfg["explain"]["permutations_gpu"],
                   cfg["xfakenet"]["highlight_threshold"])

    @property
    def clip(self):
        if self._clip is None:
            from .clip_utils import Clip
            self._clip = Clip(self.clip_name)
        return self._clip

    def __call__(self, text: str, image=None) -> dict:
        from .attribution import explain_batch, highlight
        e = self.pred.explain([text])[0]
        words, segs = [w["w"] for w in e["words"]], [s["text"] for s in e["segments"]]
        a = explain_batch(self.pred.predict_proba, [{"words": words, "segments": segs, "pred": e["pred"]}],
                          self.pred.meta["classes"], m=self.m, methods=("shap", "segment"))[0]
        shap = a.get("shap", [0.0] * len(words))
        seg_sc = a.get("segment_shap") or [s["weight"] for s in e["segments"]]
        top_seg = segs[int(np.argmax(seg_sc))] if segs else text
        out = {"label": e["pred"], "probs": e["probs"],
               "highlighted": highlight(words, shap, self.hl_thr),
               "word_scores": [{"w": w, "shap": round(float(x), 4)} for w, x in zip(words, shap)],
               "segments": [{"text": s, "shap": round(float(x), 4)} for s, x in zip(segs, seg_sc)],
               "top_segment": top_seg,
               "attention_highlighted": e["highlighted"], "attention_top_segment": e["top_segment"],
               "modality_flag": "TEXT", "cscore": None, "cscore_threshold": self.thr,
               "deception_pattern": None, "pattern_probs": None}
        if image is None:
            return out
        from .clip_utils import pair_features
        T, I = self.clip.embed([text], [image])
        c = float(1.0 - (T * I).sum())
        out["cscore"] = round(c, 4)
        if c > self.thr:
            out["modality_flag"] = "CROSS-MODAL"
            if self.deception is not None:
                pr = self.deception["model"].predict_proba(pair_features(T, I))[0]
                cls = list(self.deception["model"].classes_)
                out["pattern_probs"] = {k: round(float(v), 4) for k, v in zip(cls, pr)}
                out["deception_pattern"] = cls[int(np.argmax(pr))]
        elif self.modality is not None:
            m = self.modality
            zt = float(m["text"].decision_function(T)[0])
            zi = float(m["image"].decision_function(I)[0])
            z = m["fusion_scaler"].transform([[zt, zi]])[0]
            ct, ci = m["fusion_coef"][0] * z[0], m["fusion_coef"][1] * z[1]
            out["modality_flag"] = "TEXT" if abs(ct) >= abs(ci) else "IMAGE"
            out["modality_contrib"] = {"text": round(float(ct), 4), "image": round(float(ci), 4)}
        return out

In [ ]:
%%writefile scripts/16_deception.py
"""Week 4: six-category deception-pattern attribution for image-text pairs.

Stages (run in order; each is skipped if its inputs are missing):
  label : Gemini (multimodal) labels each pair with one of six deception patterns or
          "consistent". Pairs = all 389 real FakeNewsNet pairs + 200 shuffled (mismatched) pairs.
  train : a small classifier (logistic regression on CLIP text/image features + C-score)
          learns the six patterns from Gemini's labels; 5-fold cross-validated.
          This classifier is what the pipeline uses for CROSS-MODAL items (no API at run time).
  page  : builds results/week4/handcheck.html with 100 pairs for you to verify by hand
          (works offline in any browser; download your answers as CSV at the end).
  score : after you upload the CSV, measures Gemini's and the classifier's agreement with you.

Usage: python scripts/16_deception.py --stage all
       python scripts/16_deception.py --stage score
"""
from __future__ import annotations

import argparse
import base64
import html
import io
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed, write_json  # noqa: E402

CATS = {
    "out_of_context": "Genuine photo, but from a different event, time or story than the headline",
    "misattributed_entity": "Shows a different person, group or organisation than the headline names",
    "wrong_location_or_event": "Shows a different place or kind of event than the headline describes",
    "exaggeration": "Image and headline differ sharply in drama or emotion (one sensationalises the other)",
    "manipulated_or_synthetic": "Looks edited, composited, a meme, a screenshot with added text, or AI-generated",
    "irrelevant_stock": "Generic stock or decorative image with no real connection to the specific claim",
    "consistent": "Image plausibly matches the headline (no deception pattern)",
}
PATTERNS = [c for c in CATS if c != "consistent"]

PROMPT = """You are annotating news headline + image pairs for research on image-text deception.
For each item, choose exactly ONE category:
{cats}

Judge only the relationship between the headline and the image (not whether the claim is true).
Return a JSON array, one object per item, same order:
[{{"i": <index>, "category": "<category key>", "confidence": <0-1>, "reason": "<max 15 words>"}}, ...]
"""


def items(cfg: dict) -> pd.DataFrame:
    """Real pairs + a sample of shuffled pairs (from the C-score step)."""
    dc = cfg["deception"]
    al = pd.read_csv(p(f"{cfg['paths']['processed']}/fnn_pairs.csv"))
    al = al[al["image_path"].map(lambda x: p(x).exists())]
    al = al.assign(pair_type="aligned")[["id", "title", "image_path", "fnn_label", "site", "pair_type"]]
    parts = [al]
    cp = p(f"{cfg['cscore']['out_dir']}/cscore_pairs.csv")
    if cp.exists():
        mis = pd.read_csv(cp)
        mis = mis[mis["pair_type"] == "mismatched"]
        mis = mis.sample(min(len(mis), dc["n_mismatched"]), random_state=cfg["seed"])
        parts.append(mis[["id", "title", "image_path", "fnn_label", "site", "pair_type"]])
    df = pd.concat(parts, ignore_index=True)
    df["key"] = df["id"].astype(str) + "|" + df["pair_type"]
    return df.reset_index(drop=True)


def jpeg_bytes(path, side: int) -> bytes:
    from PIL import Image
    im = Image.open(p(path)).convert("RGB")
    im.thumbnail((side, side))
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=85)
    return buf.getvalue()


# ---------------- stage: label ----------------
def stage_label(cfg: dict, df: pd.DataFrame) -> None:
    dc = cfg["deception"]
    key = os.environ.get("GEMINI_API_KEY", "")
    if not key:
        print("[deception] no GEMINI_API_KEY in the environment; labelling skipped")
        return
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=key)
    out = p(dc["labels"])
    done = {json.loads(l)["key"] for l in open(out)} if out.exists() else set()
    todo = df[~df["key"].isin(done)]
    print(f"[deception] {len(df)} pairs, {len(done)} labelled, {len(todo)} to go", flush=True)
    cats = "\n".join(f'- "{k}": {v}' for k, v in CATS.items())
    model = None
    gcfg = types.GenerateContentConfig(response_mime_type="application/json", temperature=0.2)
    bs = dc["batch_size"]
    with open(out, "a") as f:
        for s in range(0, len(todo), bs):
            b = todo.iloc[s:s + bs]
            contents = [PROMPT.format(cats=cats)]
            for i, r in enumerate(b.itertuples(index=False)):
                contents += [f"Item {i}. Headline: {r.title}",
                             types.Part.from_bytes(data=jpeg_bytes(r.image_path, dc["image_side"]),
                                                   mime_type="image/jpeg")]
            got = None
            for m in ([model] if model else cfg["paraphrase"]["models"]):
                for attempt in range(5):
                    try:
                        resp = client.models.generate_content(model=m, contents=contents, config=gcfg)
                        got = {int(o["i"]): o for o in json.loads(resp.text)}
                        model = m
                        break
                    except Exception as e:
                        msg = str(e)
                        if "404" in msg or "NOT_FOUND" in msg or "402" in msg or "credits" in msg.lower():
                            print(f"[deception] {m}: {msg[:120]}", flush=True)
                            break
                        time.sleep(min(60, 5 * 2 ** attempt))
                if got is not None:
                    break
            if got is None:
                print(f"[deception] batch {s // bs} failed; will retry on the next run", flush=True)
                continue
            for i, r in enumerate(b.itertuples(index=False)):
                o = got.get(i, {})
                cat = str(o.get("category", "")).strip()
                f.write(json.dumps({"key": r.key, "category": cat if cat in CATS else "invalid",
                                    "confidence": o.get("confidence"), "reason": o.get("reason", ""),
                                    "model": model}) + "\n")
            f.flush()
            print(f"[deception] labelled {min(s + bs, len(todo))}/{len(todo)}", flush=True)
            time.sleep(dc["sleep_s"])


def load_labels(cfg: dict, df: pd.DataFrame) -> pd.DataFrame:
    f = p(cfg["deception"]["labels"])
    if not f.exists():
        return pd.DataFrame()
    lab = pd.DataFrame([json.loads(l) for l in open(f)]).drop_duplicates("key", keep="last")
    return df.merge(lab, on="key", how="inner")


# ---------------- stage: train ----------------
def stage_train(cfg: dict, df: pd.DataFrame) -> dict:
    import joblib
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from xfakenet.clip_utils import Clip, pair_features

    lab = load_labels(cfg, df)
    if not len(lab):
        print("[deception] no labels yet; training skipped")
        return {}
    res = {"n_labelled": len(lab),
           "label_counts": lab["category"].value_counts().to_dict(),
           "label_counts_by_pair_type": lab.groupby(["pair_type", "category"]).size().unstack(fill_value=0)
           .to_dict(orient="index")}

    feats_f = p(cfg["deception"]["features"])
    T, I = Clip(cfg["cscore"]["clip_model"]).embed(lab["title"].tolist(), [p(x) for x in lab["image_path"]])
    np.savez(feats_f, key=lab["key"].to_numpy(), T=T, I=I)
    X = pair_features(T, I)

    pat = lab["category"].isin(PATTERNS).to_numpy()
    y = lab.loc[pat, "category"].to_numpy()
    counts = pd.Series(y).value_counts()
    keep_cls = counts[counts >= cfg["deception"]["min_per_class"]].index.tolist()
    m = np.isin(y, keep_cls)
    Xp, yp = X[pat][m], y[m]
    res["classifier_classes"] = keep_cls
    res["dropped_rare_classes"] = counts[counts < cfg["deception"]["min_per_class"]].to_dict()
    if len(keep_cls) < 2:
        print("[deception] fewer than 2 pattern classes with enough examples; classifier skipped")
        write_json(res, f"{cfg['week4']['out_dir']}/deception_results.json")
        return res
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=cfg["deception"]["C"], max_iter=5000,
                                                             class_weight="balanced"))
    n_splits = int(min(5, counts[keep_cls].min()))
    cv = StratifiedKFold(n_splits, shuffle=True, random_state=cfg["seed"])
    oof = cross_val_predict(clf, Xp, yp, cv=cv)
    majority = pd.Series(yp).value_counts(normalize=True).iloc[0]
    res["classifier_cv"] = {"n": int(len(yp)), "folds": n_splits,
                            "accuracy": round(float(accuracy_score(yp, oof)), 4),
                            "macro_f1": round(float(f1_score(yp, oof, average="macro")), 4),
                            "majority_class_accuracy": round(float(majority), 4)}
    keys = lab.loc[pat, "key"].to_numpy()[m]
    pd.DataFrame({"key": keys, "gemini_category": yp, "oof_pred": oof}).to_csv(
        p(cfg["deception"]["oof"]), index=False)
    clf.fit(Xp, yp)
    joblib.dump({"model": clf, "classes": keep_cls}, p(cfg["deception"]["model"]))
    print(f"[deception] classifier CV: {res['classifier_cv']}", flush=True)
    write_json(res, f"{cfg['week4']['out_dir']}/deception_results.json")
    return res


# ---------------- stage: page ----------------
PAGE_CSS = """
:root{--bg:#f7f6f2;--card:#fff;--ink:#1c1c1a;--muted:#66655f;--line:#e2dfd6;--acc:#b45309;--ok:#2f7d4f;--bad:#b3261e}
@media (prefers-color-scheme:dark){:root{--bg:#161614;--card:#21201d;--ink:#eceae4;--muted:#a3a198;--line:#34332e;--acc:#f59e0b;--ok:#6cc58f;--bad:#f2867e}}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.5 -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif}
header{position:sticky;top:0;background:var(--bg);border-bottom:1px solid var(--line);padding:12px 16px;z-index:2}
.wrap{max-width:860px;margin:0 auto}h1{font-size:20px;margin:0}
.bar{display:flex;gap:12px;align-items:center;flex-wrap:wrap;margin-top:8px}
.prog{flex:1;min-width:160px;height:8px;background:var(--line);border-radius:4px;overflow:hidden}.prog i{display:block;height:100%;background:var(--acc)}
button{font:inherit;padding:8px 14px;border-radius:8px;border:1px solid var(--acc);background:var(--acc);color:#fff;cursor:pointer}
main{padding:16px}details{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:10px 14px;margin-bottom:16px}
details li{margin:2px 0}.card{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:14px;margin-bottom:14px;
display:grid;grid-template-columns:220px 1fr;gap:14px}.card.done{border-color:var(--ok)}
.card img{width:100%;border-radius:6px;object-fit:cover;max-height:220px}
.n{font-size:12px;color:var(--muted)}.hl{font-weight:600;margin:2px 0 8px}.gem{font-size:14px;margin-bottom:8px}
.gem b{color:var(--acc)}.why{color:var(--muted);font-size:13px}label{margin-right:14px;cursor:pointer}
select{font:inherit;margin-top:6px;padding:4px;border-radius:6px;border:1px solid var(--line);background:var(--card);color:var(--ink)}
@media (max-width:600px){.card{grid-template-columns:1fr}}
"""

PAGE_JS = """
const KEY='xfn-handcheck-v1';let S={};try{S=JSON.parse(localStorage.getItem(KEY)||'{}')}catch(e){S={}}
function save(){try{localStorage.setItem(KEY,JSON.stringify(S))}catch(e){}upd()}
function upd(){const n=ITEMS.filter(it=>S[it.key]&&(S[it.key].v==='correct'||(S[it.key].v==='wrong'&&S[it.key].c))).length;
document.getElementById('cnt').textContent=n+' / '+ITEMS.length+' checked';document.getElementById('pbar').style.width=(100*n/ITEMS.length)+'%';
ITEMS.forEach(it=>{const el=document.getElementById('c-'+it.idx);const s=S[it.key];el.classList.toggle('done',!!s&&(s.v==='correct'||(s.v==='wrong'&&!!s.c)))})}
function setV(k,v){S[k]=S[k]||{};S[k].v=v;if(v==='correct')S[k].c='';document.getElementById('s-'+k).disabled=(v!=='wrong');save()}
function setC(k,c){S[k]=S[k]||{v:'wrong'};S[k].c=c;save()}
function dl(){const rows=[['key','gemini_category','verdict','human_category']];
ITEMS.forEach(it=>{const s=S[it.key]||{};rows.push([it.key,it.gem,s.v||'',s.v==='correct'?it.gem:(s.c||'')])});
const csv=rows.map(r=>r.map(x=>'"'+String(x).replace(/"/g,'""')+'"').join(',')).join('\\n');
const a=document.createElement('a');a.href=URL.createObjectURL(new Blob([csv],{type:'text/csv'}));a.download='handcheck_results.csv';a.click()}
window.addEventListener('DOMContentLoaded',()=>{ITEMS.forEach(it=>{const s=S[it.key];if(!s)return;
const r=document.querySelector('input[name="v-'+it.idx+'"][value="'+s.v+'"]');if(r)r.checked=true;
const sel=document.getElementById('s-'+it.key);sel.disabled=s.v!=='wrong';if(s.c)sel.value=s.c});upd()});
"""


def stage_page(cfg: dict, df: pd.DataFrame) -> None:
    lab = load_labels(cfg, df)
    lab = lab[lab["category"].isin(CATS)]
    if not len(lab):
        print("[deception] no labels yet; hand-check page skipped")
        return
    n = cfg["deception"]["handcheck_n"]
    rng = np.random.default_rng(cfg["seed"])
    # at least a few of every category, the rest proportional
    picks = []
    for c, g in lab.groupby("category"):
        picks += g.sample(min(len(g), 5), random_state=cfg["seed"]).index.tolist()
    rest = lab.drop(picks)
    picks += rest.sample(min(len(rest), max(0, n - len(picks))), random_state=cfg["seed"]).index.tolist()
    sample = lab.loc[picks].sample(frac=1.0, random_state=cfg["seed"]).head(n).reset_index(drop=True)
    sample[["key", "category"]].to_csv(p(cfg["deception"]["handcheck_sample"]), index=False)

    js_items, cards = [], []
    opts = "".join(f'<option value="{k}">{k.replace("_", " ")}</option>' for k in CATS)
    for i, r in sample.iterrows():
        b64 = base64.b64encode(jpeg_bytes(r.image_path, 360)).decode()
        k = html.escape(r.key, quote=True)
        js_items.append({"idx": i, "key": r.key, "gem": r.category})
        cards.append(
            f'<div class="card" id="c-{i}"><img alt="" src="data:image/jpeg;base64,{b64}">'
            f'<div><div class="n">#{i + 1}</div><div class="hl">{html.escape(str(r.title))}</div>'
            f'<div class="gem">Gemini: <b>{r.category.replace("_", " ")}</b> '
            f'<span class="why">— {html.escape(str(r.reason))}</span></div>'
            f'<label><input type="radio" name="v-{i}" value="correct" onchange="setV(\'{k}\',\'correct\')"> Correct</label>'
            f'<label><input type="radio" name="v-{i}" value="wrong" onchange="setV(\'{k}\',\'wrong\')"> Wrong, should be:</label>'
            f'<br><select id="s-{k}" disabled onchange="setC(\'{k}\',this.value)"><option value="">choose…</option>{opts}</select>'
            f'</div></div>')
    legend = "".join(f"<li><b>{k.replace('_', ' ')}</b>: {html.escape(v)}</li>" for k, v in CATS.items())
    page = (f"<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' "
            f"content='width=device-width,initial-scale=1'><title>Deception hand-check</title><style>{PAGE_CSS}</style>"
            f"</head><body><header><div class='wrap'><h1>Hand-check Gemini's deception labels</h1><div class='bar'>"
            f"<div class='prog'><i id='pbar' style='width:0'></i></div><span id='cnt'></span>"
            f"<button onclick='dl()'>Download results (CSV)</button></div></div></header><main class='wrap'>"
            f"<details open><summary><b>How to check</b> (your answers are saved in this browser as you go)</summary>"
            f"<p>For each pair, look at the headline and image and decide whether Gemini's category is right. "
            f"If not, pick the right one. When finished, click <b>Download results (CSV)</b> and upload that file "
            f"in the Week 4 notebook.</p><ul>{legend}</ul></details>{''.join(cards)}</main>"
            f"<script>const ITEMS={json.dumps(js_items)};{PAGE_JS}</script></body></html>")
    out = p(f"{cfg['week4']['out_dir']}/handcheck.html")
    out.write_text(page, encoding="utf-8")
    print(f"[deception] hand-check page: {out} ({len(sample)} pairs)")


# ---------------- stage: score ----------------
def stage_score(cfg: dict) -> None:
    from sklearn.metrics import cohen_kappa_score
    f = p(cfg["deception"]["handcheck_results"])
    if not f.exists():
        print("[deception] no hand-check results uploaded yet; scoring skipped")
        return
    h = pd.read_csv(f)
    h = h[h["verdict"].isin(["correct", "wrong"]) & h["human_category"].fillna("").astype(str).str.len().gt(0)]
    resf = p(f"{cfg['week4']['out_dir']}/deception_results.json")
    res = json.load(open(resf)) if resf.exists() else {}
    hc = {"n_checked": int(len(h)),
          "gemini_accuracy": round(float((h["verdict"] == "correct").mean()), 4) if len(h) else None,
          "gemini_kappa": round(float(cohen_kappa_score(h["gemini_category"], h["human_category"])), 4)
          if len(h) > 1 else None,
          "pattern_vs_consistent_agreement": round(float(
              ((h["gemini_category"] == "consistent") == (h["human_category"] == "consistent")).mean()), 4)
          if len(h) else None}
    oof = p(cfg["deception"]["oof"])
    if oof.exists():
        o = pd.read_csv(oof).merge(h[["key", "human_category"]], on="key")
        if len(o):
            hc["classifier_accuracy_vs_human"] = round(float((o["oof_pred"] == o["human_category"]).mean()), 4)
            hc["classifier_n"] = int(len(o))
    res["handcheck"] = hc
    write_json(res, resf)
    print(f"[deception] hand-check: {hc}")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", default="all", choices=["all", "label", "train", "page", "score"])
    args = ap.parse_args()
    cfg = load_config()
    set_seed(cfg["seed"])
    df = items(cfg)
    if args.stage in ("all", "label"):
        stage_label(cfg, df)
    if args.stage in ("all", "train"):
        stage_train(cfg, df)
    if args.stage in ("all", "page"):
        stage_page(cfg, df)
    if args.stage in ("all", "score"):
        stage_score(cfg)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/17_pipeline_demo.py
"""Week 4: fit the deployable modality models and run the integrated pipeline on example inputs.

1. Fits text-only and image-only logistic models (CLIP features, all 389 FakeNewsNet pairs)
   and the fusion weights used for the TEXT / IMAGE flag; saves runs/modality.joblib.
2. Runs XFakeNetPipeline on 5 LIAR statements (text only) and 6 FakeNewsNet items
   (headline + image; 3 above and 3 below the C-score threshold).

Output: runs/modality.joblib, results/week4/pipeline_demo.json, results/week4/pipeline_demo.md
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed, write_json  # noqa: E402


def fit_modality(cfg: dict, df: pd.DataFrame, T: np.ndarray, I: np.ndarray) -> dict:
    import joblib
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    y = (df["fnn_label"] == "fake").astype(int).to_numpy()
    C = cfg["modality"]["C"]
    mk = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))  # noqa: E731
    # out-of-fold logits for the fusion weights (as in script 14), final models on all data
    folds = list(StratifiedKFold(5, shuffle=True, random_state=cfg["seed"]).split(T, df["site"] + df["fnn_label"]))
    zt, zi = np.zeros(len(y)), np.zeros(len(y))
    for tr, te in folds:
        zt[te] = mk().fit(T[tr], y[tr]).decision_function(T[te])
        zi[te] = mk().fit(I[tr], y[tr]).decision_function(I[te])
    sc = StandardScaler().fit(np.column_stack([zt, zi]))
    fus = LogisticRegression(max_iter=5000).fit(sc.transform(np.column_stack([zt, zi])), y)
    art = {"text": mk().fit(T, y), "image": mk().fit(I, y), "fusion_scaler": sc, "fusion_coef": fus.coef_[0].tolist()}
    joblib.dump(art, p(cfg["pipeline"]["modality_model"]))
    return art


def main() -> None:
    from xfakenet.clip_utils import Clip
    from xfakenet.pipeline import XFakeNetPipeline
    cfg = load_config()
    set_seed(cfg["seed"])
    out_dir = cfg["week4"]["out_dir"]

    df = pd.read_csv(p(f"{cfg['paths']['processed']}/fnn_pairs.csv"))
    df = df[df["image_path"].map(lambda x: p(x).exists())].reset_index(drop=True)
    T, I = Clip(cfg["cscore"]["clip_model"]).embed(df["title"].tolist(), [p(x) for x in df["image_path"]])
    fit_modality(cfg, df, T, I)
    print("[pipeline] modality models saved", flush=True)

    pipe = XFakeNetPipeline.load()
    test = pd.read_csv(p(f"{cfg['paths']['processed']}/corpus_test.csv"))
    rows = []
    sample = pd.concat([g.sample(2, random_state=cfg["seed"]) for _, g in test.groupby("label")]).head(5)
    for r in sample.itertuples(index=False):
        o = pipe(r.text)
        rows.append({"input": "text", "text": r.text, "gold": r.label, **{k: o[k] for k in
                     ("label", "probs", "highlighted", "top_segment", "modality_flag", "cscore",
                      "deception_pattern", "attention_highlighted")}})
    c = 1.0 - (T * I).sum(1)
    above = np.where(c > pipe.thr)[0][:3]
    below = np.where(c <= pipe.thr)[0][:3]
    for k in list(above) + list(below):
        r = df.iloc[k]
        o = pipe(r["title"], image=str(p(r["image_path"])))
        rows.append({"input": "text+image", "text": r["title"], "gold": f"FakeNewsNet {r['fnn_label']}",
                     "image": r["image_path"], **{k2: o[k2] for k2 in
                     ("label", "probs", "highlighted", "top_segment", "modality_flag", "cscore",
                      "deception_pattern")}})
    write_json(rows, f"{out_dir}/pipeline_demo.json")
    L = ["| Input | Text | Gold | Predicted | Highlighted | Modality | C-score | Pattern |",
         "|---|---|---|---|---|---|---|---|"]
    for r in rows:
        t = r["text"] if len(r["text"]) < 90 else r["text"][:87] + "…"
        L.append(f"| {r['input']} | {t} | {r['gold']} | {r['label']} | {', '.join(r['highlighted'][:4])} | "
                 f"{r['modality_flag']} | {'' if r['cscore'] is None else r['cscore']} | "
                 f"{(r['deception_pattern'] or '').replace('_', ' ')} |")
    open(p(f"{out_dir}/pipeline_demo.md"), "w").write("\n".join(L))
    print("\n".join(L))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/18_final_tables.py
"""Week 4: final results tables (classification, robustness, ablations, C-score, faithfulness, attribution).

Seeds: runs trained with seed 42 (weeks 1-3) plus any extra-seed runs named <run>_s<seed>
(e.g. xfakenet_v2_s1). With 2+ seeds, tables show mean ± standard deviation.

Output: results/final/results_tables.md, results/final/results_tables.json
"""
from __future__ import annotations

import glob
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, write_json  # noqa: E402

FAMILIES = {  # name: (metrics of seed-42 run, glob for extra seeds, fresh-attack files)
    "Baseline (RoBERTa)": ("runs/baseline/metrics.json", "runs/baseline_s*/metrics.json",
                           ["data/augment/textfooler_test.jsonl", "data/augment/textfooler_test_baseline_s*.jsonl"]),
    "XFakeNet, no augmentation": ("runs/xfakenet_noaug/metrics.json", "runs/xfakenet_noaug_s*/metrics.json",
                                  ["data/augment/textfooler_test_xfakenet_noaug.jsonl",
                                   "data/augment/textfooler_test_xfakenet_noaug_s*.jsonl"]),
    "XFakeNet v2 (augmented)": ("runs/xfakenet_v2/metrics.json", "runs/xfakenet_v2_s*/metrics.json",
                                ["data/augment/textfooler_test_xfakenet_v2.jsonl",
                                 "data/augment/textfooler_test_xfakenet_v2_s*.jsonl"]),
    "XFakeNet v3 (2nd adversarial round)": ("runs/xfakenet_v3/metrics.json", "runs/xfakenet_v3_s*/metrics.json",
                                            ["data/augment/textfooler_test_xfakenet_v3.jsonl"]),
}


def jload(f):
    f = p(f)
    return json.load(open(f)) if f.exists() else None


def ms(xs: list[float], d: int = 3) -> str:
    xs = [x for x in xs if x is not None]
    if not xs:
        return "–"
    if len(xs) == 1:
        return f"{xs[0]:.{d}f}"
    return f"{np.mean(xs):.{d}f} ± {np.std(xs, ddof=1):.{d}f}"


def attack_stats(f: str) -> dict | None:
    f = p(f)
    if not f.exists():
        return None
    t = pd.DataFrame([json.loads(l) for l in open(f)])
    vc = t["status"].value_counts()
    s, fl, n = int(vc.get("success", 0)), int(vc.get("failed", 0)), len(t)
    return {"acc_under_attack": fl / n, "success_rate": s / max(1, s + fl)}


def main() -> None:
    cfg = load_config()
    out = {}
    L = ["# XFakeNet – final results tables", ""]

    # ---- T1 classification + T2 robustness
    t1 = ["## Table 1. Classification on the clean LIAR test set (1,283 statements, 3 classes)", "",
          "| Model | Seeds | Accuracy | Macro-F1 | F1 Fabricated | F1 Misleading | F1 Credible |",
          "|---|---|---|---|---|---|---|"]
    t2 = ["## Table 2. Robustness (each model attacked by its own TextFooler run, 499 test statements)", "",
          "| Model | Seeds | Accuracy under attack | Attack success rate | Paraphrased: prediction unchanged |",
          "|---|---|---|---|---|"]
    rob = jload(f"{cfg['week3']['out_dir']}/robustness.json") or {}
    for name, (m42, extra, atk) in FAMILIES.items():
        runs = [m for m in [jload(m42)] + [jload(f) for f in sorted(glob.glob(str(p(extra))))] if m]
        if not runs:
            continue
        get = lambda k, c=None: [r["test"]["per_class"][c]["f1-score"] if c else r["test"][k] for r in runs]  # noqa
        t1.append(f"| {name} | {len(runs)} | {ms(get('accuracy'))} | {ms(get('macro_f1'))} | "
                  f"{ms(get(None, 'Fabricated'))} | {ms(get(None, 'Misleading'))} | {ms(get(None, 'Credible'))} |")
        atks = [a for f in atk for g in sorted(glob.glob(str(p(f)))) if (a := attack_stats(g))]
        para = rob.get(name, {}).get("paraphrase", {}).get("prediction_unchanged")
        if atks:
            t2.append(f"| {name} | {len(atks)} | {ms([a['acc_under_attack'] for a in atks])} | "
                      f"{ms([a['success_rate'] for a in atks])} | {'' if para is None else f'{para*100:.0f}%'} |")
        out[name] = {"n_seeds": len(runs), "macro_f1": get("macro_f1"), "accuracy": get("accuracy"),
                     "acc_under_attack": [a["acc_under_attack"] for a in atks],
                     "attack_success": [a["success_rate"] for a in atks]}
    L += t1 + ["", "*Mean ± standard deviation over seeds when more than one seed was run.*", ""] + t2 + [""]

    # ---- T3 ablations
    L += ["## Table 3. Ablations", ""]
    a, b = out.get("XFakeNet, no augmentation"), out.get("XFakeNet v2 (augmented)")
    if a and b:
        L += ["**Adversarial + paraphrase augmentation** (XFakeNet without vs with):", "",
              "| | Without | With |", "|---|---|---|",
              f"| Clean macro-F1 | {ms(a['macro_f1'])} | {ms(b['macro_f1'])} |",
              f"| Accuracy under attack | {ms(a['acc_under_attack'])} | {ms(b['acc_under_attack'])} |",
              f"| Attack success rate | {ms(a['attack_success'])} | {ms(b['attack_success'])} |", ""]
    mres = jload(f"{cfg['week3']['out_dir']}/modality_results.json")
    if mres:
        au = mres["fake_vs_real_auc_5fold"]
        L += ["**C-score signal** (FakeNewsNet fake vs real, 5-fold ROC-AUC):", "",
              "| Features | AUC |", "|---|---|"] + [f"| {k} | {v:.3f} |" for k, v in au.items()] + [""]

    # ---- T4 C-score calibration
    c = jload(f"{cfg['cscore']['out_dir']}/cscore_results.json")
    if c:
        L += ["## Table 4. C-score calibration (aligned vs mismatched pairs, test split)", "",
              f"ROC-AUC {c['test_auc_aligned_vs_mismatched']:.3f}", "",
              "| Threshold | Value | Accuracy | Flags genuine pairs | Flags mismatched pairs |", "|---|---|---|---|---|"]
        for k, r in c["thresholds"].items():
            L.append(f"| {k} | {r['threshold']:.3f} | {r['accuracy']:.3f} | {r['flag_rate_aligned']*100:.0f}% | "
                     f"{r['flag_rate_mismatched']*100:.0f}% |")
        L.append("")

    # ---- T5 faithfulness
    fm = p(f"{cfg['week4']['out_dir']}/faithfulness.md")
    if fm.exists():
        L += ["## Table 5. Explanation faithfulness (deletion test) and agreement with SHAP", "",
              fm.read_text(), ""]

    fe = p(f"{cfg['week4']['out_dir']}/faithful_explanations.md")
    if fe.exists():
        L += ["## Table 5b. Faithful explanations: attention vs occlusion vs SHAP", "", fe.read_text(), ""]

    # ---- T6 deception attribution
    d = jload(f"{cfg['week4']['out_dir']}/deception_results.json")
    if d:
        L += ["## Table 6. Deception-pattern attribution", "",
              f"- Gemini labels: {d.get('n_labelled')} pairs; categories: {d.get('label_counts')}"]
        if "classifier_cv" in d:
            cv = d["classifier_cv"]
            L.append(f"- Classifier (CLIP features, {cv['folds']}-fold CV vs Gemini labels, {cv['n']} pairs): "
                     f"accuracy {cv['accuracy']:.3f}, macro-F1 {cv['macro_f1']:.3f} "
                     f"(majority-class accuracy {cv['majority_class_accuracy']:.3f})")
        if "handcheck" in d:
            h = d["handcheck"]
            L.append(f"- Hand-check of {h['n_checked']} pairs: Gemini correct {h['gemini_accuracy']*100:.0f}%, "
                     f"Cohen's kappa {h['gemini_kappa']}"
                     + (f"; classifier agrees with human labels {h['classifier_accuracy_vs_human']*100:.0f}% "
                        f"(n={h['classifier_n']})" if "classifier_accuracy_vs_human" in h else ""))
        else:
            L.append("- Hand-check: not uploaded yet")
        L.append("")

    md = "\n".join(L)
    open(p("results/final/results_tables.md"), "w").write(md)
    write_json(out, "results/final/results_tables.json")
    print(md)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/19_faithful_explanations.py
"""Week 4 fix: faithful explanations (SHAP / occlusion) vs the built-in attention, same tests.

For XFakeNet v2 on LIAR test statements:
  * word scores from: attention (built-in), occlusion (leave-one-out), SHAP (permutation Shapley)
  * segment scores from: attention (built-in), exact Shapley over segments
Faithfulness tests (probability of the predicted class):
  * comprehensiveness: remove the top-k words (k = 1, 3) -> how much does it drop? (higher = better)
  * sufficiency: keep ONLY the top-3 words -> how much does it drop? (lower = better)
  * each vs the same number of random words (3 draws); "beats random" = % of statements
  * segments: remove the top segment by each method vs a random segment
Note: occlusion at k = 1 is the comprehensiveness test itself, so that cell is not informative.

GPU: all 1,283 test statements, 10 permutations. CPU only: 400 statements, 6 permutations.

Output: results/week4/faithful_explanations.json / .md, results/week4/explanations_shap.html,
        results/week4/explanations_shap_test.jsonl
"""
from __future__ import annotations

import html
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.attribution import explain_batch, highlight  # noqa: E402
from xfakenet.common import load_config, p, write_json  # noqa: E402

METHODS = ["attention", "occlusion", "shap"]


def evaluate(predict, rows: list[dict], classes: list[str], rng: np.random.Generator) -> dict:
    texts, meta = [], []
    for i, r in enumerate(rows):
        w, n = r["words"], len(r["words"])
        if n < 5:
            continue
        for meth in METHODS:
            order = list(np.argsort(-np.asarray(r[meth])))
            for k in (1, 3):
                texts.append(" ".join(x for j, x in enumerate(w) if j not in set(order[:k])))
                meta.append((i, meth, f"remove {k}"))
            texts.append(" ".join(x for j, x in enumerate(w) if j in set(order[:3])))
            meta.append((i, meth, "keep only 3"))
        for _ in range(3):
            for k in (1, 3):
                rnd = set(rng.choice(n, k, replace=False).tolist())
                texts.append(" ".join(x for j, x in enumerate(w) if j not in rnd))
                meta.append((i, "random", f"remove {k}"))
            keep = set(rng.choice(n, 3, replace=False).tolist())
            texts.append(" ".join(x for j, x in enumerate(w) if j in keep))
            meta.append((i, "random", "keep only 3"))
        segs = r["segments"]
        if 2 <= len(segs) <= 8 and r.get("segment_shap") is not None:
            for meth, sc in [("attention", r["seg_attention"]), ("shap", r["segment_shap"])]:
                top = int(np.argmax(sc))
                texts.append(" ".join(s for j, s in enumerate(segs) if j != top))
                meta.append((i, meth, "remove top segment"))
            other = int(rng.integers(len(segs)))
            texts.append(" ".join(s for j, s in enumerate(segs) if j != other))
            meta.append((i, "random", "remove top segment"))
    probs = predict(texts)
    df = pd.DataFrame([{"i": i, "method": m, "test": t,
                        "drop": rows[i]["p0"] - pr[classes.index(rows[i]["pred"])]}
                       for (i, m, t), pr in zip(meta, probs)])
    res = {}
    for test, g in df.groupby("test"):
        rnd = g[g.method == "random"].groupby("i")["drop"].mean()
        res[test] = {}
        for meth in METHODS + ["random"]:
            x = g[g.method == meth]
            if not len(x):
                continue
            per = x.groupby("i")["drop"].mean()
            better = (per < rnd.reindex(per.index)) if test == "keep only 3" else (per > rnd.reindex(per.index))
            res[test][meth] = {"n": int(per.size), "mean_drop": round(float(per.mean()), 4),
                               "beats_random_pct": None if meth == "random" else round(float(better.mean() * 100), 1)}
    return res


def render(rows, classes, thr):
    def line(words, scores, color):
        s = np.asarray(scores, dtype=float)
        top = s.max() if len(s) and s.max() > 0 else 1.0
        out = []
        for w, x in zip(words, s):
            a = max(0.0, x / top)
            cls = "hl" if a >= thr else ""
            out.append(f'<span class="{cls}" style="background:rgba({color},{a*0.55:.2f})">{html.escape(w)}</span>')
        return " ".join(out)
    cards = []
    for r in rows:
        ok = r["pred"] == r["label"]
        cards.append(
            f'<div class="card"><div class="meta">Gold <b>{r["label"]}</b> · Predicted '
            f'<b class="{"ok" if ok else "bad"}">{r["pred"]}</b> ({r["p0"]:.2f})</div>'
            f'<div class="row"><span class="lab">Attention</span><div>{line(r["words"], r["attention"], "120,120,120")}</div></div>'
            f'<div class="row"><span class="lab">SHAP</span><div>{line(r["words"], r["shap"], "217,119,6")}</div></div></div>')
    css = (":root{--bg:#fbfaf7;--ink:#1d1d1b;--muted:#6b6a65;--line:#e4e1d8}*{box-sizing:border-box}"
           "body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.6 -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif}"
           "main{max-width:920px;margin:0 auto;padding:28px 16px 60px}h1{font-size:22px;margin:0 0 4px}p{color:var(--muted)}"
           ".card{background:#fff;border:1px solid var(--line);border-radius:10px;padding:14px 16px;margin:0 0 12px}"
           ".meta{font-size:13px;color:var(--muted);margin-bottom:8px}.ok{color:#2f7d4f}.bad{color:#b3261e}"
           ".row{display:grid;grid-template-columns:84px 1fr;gap:10px;margin:4px 0}.lab{font-size:12px;color:var(--muted);padding-top:3px}"
           "span{padding:1px 2px;border-radius:3px}span.hl{outline:1.5px solid #b45309;font-weight:600}")
    return (f"<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,"
            f"initial-scale=1'><title>Attention vs SHAP</title><style>{css}</style></head><body><main>"
            f"<h1>Built-in attention vs SHAP explanations</h1><p>XFakeNet v2 on LIAR test statements. Shading = how much a "
            f"word supports the predicted class; outlined = highlighted (≥ {thr} of the strongest word).</p>"
            f"{''.join(cards)}</main></body></html>")


def main() -> None:
    import torch
    from xfakenet.model import Predictor
    cfg = load_config()
    ec = cfg["explain"]
    out_dir = cfg["week4"]["out_dir"]
    thr = cfg["xfakenet"]["highlight_threshold"]
    gpu = torch.cuda.is_available()
    m = ec["permutations_gpu"] if gpu else ec["permutations_cpu"]
    test = pd.read_csv(p(f"{cfg['paths']['processed']}/corpus_test.csv"))
    if not gpu:
        test = test.sample(min(len(test), ec["cpu_statements"]), random_state=cfg["seed"])
    print(f"[faith2] device={'GPU' if gpu else 'CPU'} statements={len(test)} permutations={m}", flush=True)

    pred = Predictor(p("runs/xfakenet_v2"))
    classes = pred.meta["classes"]
    ex = pred.explain(test["text"].tolist(), thr)
    rows = []
    for e, t in zip(ex, test.itertuples(index=False)):
        rows.append({"id": str(t.id), "text": t.text, "label": t.label, "pred": e["pred"],
                     "p0": e["probs"][e["pred"]], "words": [w["w"] for w in e["words"]],
                     "attention": [w["weight"] for w in e["words"]],
                     "segments": [s["text"] for s in e["segments"]],
                     "seg_attention": [s["weight"] for s in e["segments"]]})
    for s in range(0, len(rows), ec["chunk"]):
        part = rows[s:s + ec["chunk"]]
        for r, a in zip(part, explain_batch(pred.predict_proba, part, classes, m=m, seed=cfg["seed"] + s)):
            r["shap"], r["occlusion"] = a.get("shap", [0.0] * len(r["words"])), a.get("occlusion", [0.0] * len(r["words"]))
            r["segment_shap"] = a.get("segment_shap")
        print(f"[faith2] explained {min(s + ec['chunk'], len(rows))}/{len(rows)}", flush=True)

    res = evaluate(pred.predict_proba, rows, classes, np.random.default_rng(cfg["seed"]))
    multi = [r for r in rows if r.get("segment_shap") is not None]
    agree = float(np.mean([np.argmax(r["seg_attention"]) == np.argmax(r["segment_shap"]) for r in multi])) if multi else None
    out = {"device": "GPU" if gpu else "CPU", "statements": len(rows), "permutations": m, "tests": res,
           "attention_vs_shap_top_segment_agreement": None if agree is None else round(agree * 100, 1),
           "mean_highlighted_shap": round(float(np.mean([len(highlight(r["words"], r["shap"], thr)) for r in rows])), 2)}
    write_json(out, f"{out_dir}/faithful_explanations.json")

    names = {"attention": "Attention (built-in)", "occlusion": "Occlusion", "shap": "SHAP", "random": "Random"}
    L = [f"Faithfulness of word and segment explanations, XFakeNet v2 ({len(rows)} LIAR test statements, "
         f"{'GPU' if gpu else 'CPU'}, {m} permutations). Drop = fall in the predicted-class probability.", "",
         "| Test | " + " | ".join(names[k] for k in METHODS + ["random"]) + " |",
         "|---|" + "---|" * (len(METHODS) + 1)]
    for test in ["remove 1", "remove 3", "keep only 3", "remove top segment"]:
        if test not in res:
            continue
        cells = []
        for meth in METHODS + ["random"]:
            r = res[test].get(meth)
            if not r:
                cells.append("–")
                continue
            b = "" if r["beats_random_pct"] is None else f" (beats random {r['beats_random_pct']:.0f}%)"
            cells.append(f"{r['mean_drop']:.3f}{b}")
        lab = {"remove 1": "Remove top word ↑", "remove 3": "Remove top 3 words ↑",
               "keep only 3": "Keep only top 3 words ↓", "remove top segment": "Remove top segment ↑"}[test]
        L.append(f"| {lab} | " + " | ".join(cells) + " |")
    L += ["", "↑ higher is better, ↓ lower is better. Occlusion at \"remove top word\" is the test itself (circular).",
          f"Attention and SHAP pick the same top segment in {out['attention_vs_shap_top_segment_agreement']}% of "
          f"{len(multi)} multi-segment statements."]
    open(p(f"{out_dir}/faithful_explanations.md"), "w").write("\n".join(L))
    print("\n".join(L))

    with open(p(f"{out_dir}/explanations_shap_test.jsonl"), "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    rng = np.random.default_rng(cfg["seed"])
    pool = [r for r in rows if 8 <= len(r["words"]) <= 30]
    pick = [pool[i] for i in rng.choice(len(pool), min(25, len(pool)), replace=False)] if pool else []
    open(p(f"{out_dir}/explanations_shap.html"), "w").write(render(pick, classes, thr))


if __name__ == "__main__":
    main()

## A. Attention vs occlusion vs SHAP

In [ ]:
!python scripts/19_faithful_explanations.py 2>&1 | grep --line-buffered -E "^\[faith2\]|Error|Traceback"
from IPython.display import Markdown, HTML, display
display(Markdown(open("results/week4/faithful_explanations.md").read()))
display(HTML(open("results/week4/explanations_shap.html").read()))

## B. Pipeline examples with SHAP highlights

In [ ]:
!python scripts/17_pipeline_demo.py 2>&1 | grep -v "^\[pipeline\]" | grep -v Warning

## C. Optional: upload your hand-check answers
If you've finished the hand-check page and downloaded `handcheck_results.csv`, set the line below to `True` and run this cell.

In [ ]:
I_HAVE_FINISHED_THE_HANDCHECK = False
if I_HAVE_FINISHED_THE_HANDCHECK:
    from google.colab import files
    import shutil
    up = files.upload()
    name = next(n for n in up if n.endswith(".csv"))
    shutil.move(name, "data/deception/handcheck_results.csv")
!python scripts/16_deception.py --stage score 2>&1 | grep -E "^\[deception\]|Error"

## D. Final tables + results zip

In [ ]:
!python scripts/18_final_tables.py > /dev/null 2>&1
from IPython.display import Markdown, display
display(Markdown(open("results/final/results_tables.md").read()))
import os, shutil
os.makedirs("/content/w4b", exist_ok=True)
for src in ["results/final/results_tables.md", "results/week4/faithful_explanations.md",
            "results/week4/faithful_explanations.json", "results/week4/explanations_shap.html",
            "results/week4/pipeline_demo.md", "results/week4/pipeline_demo.json",
            "results/week4/deception_results.json"]:
    if os.path.exists(src): shutil.copy(src, "/content/w4b/" + os.path.basename(src))
shutil.make_archive("/content/xfakenet_week4b_results", "zip", "/content/w4b")
shutil.copy("/content/xfakenet_week4b_results.zip", "results/final/xfakenet_week4b_results.zip")
from google.colab import drive
drive.flush_and_unmount()
print("Saved. Download xfakenet_week4b_results.zip from Google Drive: My Drive → xfakenet → results → final.")
from google.colab import files
try:
    files.download("/content/xfakenet_week4b_results.zip")
except Exception as e:
    print("Browser download failed; use the copy on Google Drive.")